# סריקה מלאה מול האדם באמצעות Cas-OFFinder

הסריקה הפשוטה במחברת הדמו מיועדת לרצף קטן בלבד. GRCh38 הוא כ־3.1 מיליארד בסיסים, ולכן משתמשים במנוע ייעודי.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from viral_safe_target import write_cas_offinder_input, read_cas_offinder_output

## 1. טען את רשימת המועמדים

אפשר להשתמש ב־CSV מהדמו, ובהמשך להחליף אותו ב־CSV מהנתונים האמיתיים.

In [ ]:
candidate_csv = ROOT / "reports/demo_notebook/candidates.csv"
if candidate_csv.exists():
    candidates = pd.read_csv(candidate_csv)
else:
    candidates = pd.DataFrame(columns=["candidate_id", "guide_sequence"])
    print("הרץ קודם את מחברת הדמו או שנה את הנתיב.")

candidates.head()

## 2. הפק קובץ קלט

`human_fasta_directory` צריך להכיל את קובצי הכרומוזומים של GRCh38 בפורמט FASTA.

In [ ]:
human_fasta_directory = ROOT / "data/raw/GRCh38_fasta"
cas_input = ROOT / "data/processed/cas_offinder_input.txt"

if not candidates.empty:
    write_cas_offinder_input(
        candidates,
        human_fasta_directory=human_fasta_directory,
        output_path=cas_input,
        max_mismatches=3,
    )
    print(cas_input.read_text()[:600])

## 3. הרצה חיצונית

לפי התיעוד הרשמי, Cas-OFFinder מקבל קובץ קלט, סוג מכשיר (`C` למעבד או `G` ל־GPU) וקובץ פלט:

```bash
cas-offinder data/processed/cas_offinder_input.txt C data/processed/cas_offinder_output.tsv
```

הגרסה 3 מסומנת ב־repository כלא מוכנה ל־production; למחקר רציני יש לתעד במפורש איזו גרסה שימשה.

## 4. קריאת הפלט

In [ ]:
cas_output = ROOT / "data/processed/cas_offinder_output.tsv"
if cas_output.exists():
    hits = read_cas_offinder_output(cas_output)
    display(hits.head())
    summary = hits.groupby("candidate_id").agg(
        total_hits=("candidate_id", "size"),
        min_mismatches=("mismatches", "min"),
        exact_hits=("mismatches", lambda x: int((x == 0).sum())),
    ).reset_index()
    display(summary.sort_values(["exact_hits", "min_mismatches", "total_hits"]))
else:
    print("אין פלט עדיין. הרץ את Cas-OFFinder מחוץ למחברת.")

## פירוש זהיר

- התאמה חזויה אינה הוכחה שהחיתוך יתרחש.
- היעדר hit בסף שנבחר אינו הוכחת בטיחות.
- יש להתחשב בווריאנטים אנושיים, chromatin, סוג התא ושיטת delivery.
- תוצאות קריטיות דורשות שיטות off-target ניסוייות ולא רק חיפוש מחשב.